In [3]:
import gc
import json
import os
import re
import sqlite3
import time
import tiktoken

from IPython.display import clear_output
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

from tuutrag.data import D
from tuutrag.types import BatchRequest
from tuutrag.prompt import create_batch_request
from tuutrag.prompts.manager import load_prompt
from tuutrag.utils import count_batch_request_tokens

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"
SCHEME: str        = "o200k_base"

# OpenAI Batch API limits
MAX_TOKENS_PER_BATCH: int = 1_000_000   # token ceiling per batch file
POLL_INTERVAL: int        = 30          # seconds between status checks
RETRY_WAIT: int           = 120         # seconds before resubmit on token-limit fail
MAX_RETRIES: int          = 5           # per-batch retry ceiling

# Paths
DB_PATH: Path        = D.processed / "master.db"
STAGING_DIR: Path    = D.batch_staging / "entities"
METADATA_PATH: Path  = STAGING_DIR / "batch_metadata.json"

STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Tokenizer
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

# OpenAI client
openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

print(f"DB:      {DB_PATH}")
print(f"Staging: {STAGING_DIR}")

DB:      /Users/pablobedolla/DlrowSreckah/tuutrag-open/data/processed/master.db
Staging: /Users/pablobedolla/DlrowSreckah/tuutrag-open/data/api/batch_staging/entities


In [6]:
conn = sqlite3.connect(str(DB_PATH))

total_leafs: int    = conn.execute("SELECT COUNT(*) FROM leafs").fetchone()[0]
done_leafs: int     = conn.execute("SELECT COUNT(*) FROM leafs WHERE entities IS NOT NULL").fetchone()[0]
pending_leafs: int  = total_leafs - done_leafs

print(f"Total leafs       : {total_leafs:,}")
print(f"Already processed : {done_leafs:,}")
print(f"Pending           : {pending_leafs:,}")

conn.close()

Total leafs       : 322,120
Already processed : 0
Pending           : 322,120


In [7]:
SQL_PAGE: int = 5_000   # rows fetched from SQLite at a time

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

cur = conn.cursor()
cur.execute("""
    SELECT uuid, text
    FROM   leafs
    WHERE  entities IS NULL
    ORDER  BY uuid
""")

batch_idx: int   = 0
batch_tokens: int = 0
batch_file = None
batch_count: int  = 0        # requests in the current batch file
file_entries: list[dict] = []  # metadata for all batch files
total_requests: int = 0

def _open_new_batch():
    """Open a fresh .jsonl batch file and reset counters."""
    global batch_idx, batch_tokens, batch_file, batch_count
    fname = f"entity_batch_{batch_idx}.jsonl"
    batch_file = open(STAGING_DIR / fname, "w", encoding="utf-8")
    batch_tokens = 0
    batch_count = 0
    return fname

def _close_batch(fname: str):
    """Close the current batch file and record metadata."""
    global batch_file
    batch_file.close()
    batch_file = None
    file_entries.append({
        "file_name": fname,
        "requests": batch_count,
        "tokens": batch_tokens,
    })

current_fname: str = _open_new_batch()

while True:
    rows = cur.fetchmany(SQL_PAGE)
    if not rows:
        break

    for row in rows:
        leaf_uuid: str = row["uuid"]
        leaf_text: str  = row["text"]

        prompt = load_prompt("entity.j2", text=leaf_text)
        req: BatchRequest = create_batch_request(
            custom_id=leaf_uuid,
            model=OPENAI_MODEL,
            **prompt,
        )
        req_tokens: int = count_batch_request_tokens(enc=enc, payload=req)

        # Would this request overflow the current batch?
        if batch_tokens + req_tokens > MAX_TOKENS_PER_BATCH and batch_count > 0:
            _close_batch(current_fname)
            batch_idx += 1
            current_fname = _open_new_batch()

        batch_file.write(json.dumps(req) + "\n")
        batch_tokens += req_tokens
        batch_count += 1
        total_requests += 1

    del rows
    gc.collect()

# Close the last batch file
if batch_count > 0:
    _close_batch(current_fname)
elif batch_file:
    batch_file.close()
    (STAGING_DIR / current_fname).unlink(missing_ok=True)

cur.close()
conn.close()

print(f"\nTotal pending requests : {total_requests:,}")
print(f"Batch files created   : {len(file_entries):,}")
for i, entry in enumerate(file_entries):
    print(f"  Batch {i}: {entry['requests']:,} requests | {entry['tokens']:,} tokens | {entry['file_name']}")


Total pending requests : 322,120
Batch files created   : 177
  Batch 0: 1,934 requests | 999,907 tokens | entity_batch_0.jsonl
  Batch 1: 1,928 requests | 999,656 tokens | entity_batch_1.jsonl
  Batch 2: 1,807 requests | 999,647 tokens | entity_batch_2.jsonl
  Batch 3: 1,661 requests | 999,462 tokens | entity_batch_3.jsonl
  Batch 4: 1,606 requests | 999,852 tokens | entity_batch_4.jsonl
  Batch 5: 1,707 requests | 999,730 tokens | entity_batch_5.jsonl
  Batch 6: 1,918 requests | 999,706 tokens | entity_batch_6.jsonl
  Batch 7: 1,656 requests | 999,591 tokens | entity_batch_7.jsonl
  Batch 8: 1,635 requests | 999,710 tokens | entity_batch_8.jsonl
  Batch 9: 1,623 requests | 999,492 tokens | entity_batch_9.jsonl
  Batch 10: 1,777 requests | 999,887 tokens | entity_batch_10.jsonl
  Batch 11: 1,940 requests | 999,746 tokens | entity_batch_11.jsonl
  Batch 12: 1,926 requests | 999,968 tokens | entity_batch_12.jsonl
  Batch 13: 1,770 requests | 999,921 tokens | entity_batch_13.jsonl
  Batc

In [ ]:
batch_metadata: list[dict] = []

for entry in file_entries:
    fpath = STAGING_DIR / entry["file_name"]
    with open(fpath, "rb") as f:
        uploaded = openai_client.files.create(file=f, purpose="batch")

    batch_metadata.append({
        "file_name":     entry["file_name"],
        "input_file_id": uploaded.id,
        "requests":      entry["requests"],
        "tokens":        entry["tokens"],
    })
    print(f"Uploaded {entry['file_name']} → {uploaded.id}")

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(batch_metadata, f, indent=2)

print(f"\n{len(batch_metadata):,} files uploaded.  Metadata saved to {METADATA_PATH.name}")

In [ ]:
def save_metadata(metadata: list[dict]) -> None:
    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

def get_metadata() -> list[dict]:
    if not METADATA_PATH.exists():
        return []
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def remove_from_metadata(batch_id: str) -> None:
    md = get_metadata()
    save_metadata([m for m in md if m.get("id") != batch_id])

In [ ]:
def write_entities_to_db(output_text: str) -> int:
    """
    Parse the <|> delimited entity output from OpenAI batch responses
    and UPDATE the leafs table in master.db.

    Returns the number of leafs updated.
    """
    responses: list[dict] = [
        json.loads(line)
        for line in output_text.strip().split("\n")
        if line.strip()
    ]

    conn = sqlite3.connect(str(DB_PATH))
    updated: int = 0

    buf: list[tuple[str, str]] = []

    for response in responses:
        leaf_uuid: str = response["custom_id"]
        content: str   = response["response"]["body"]["choices"][0]["message"]["content"]

        # Parse "<|>EntityOne<|><|>EntityTwo<|>" → ["EntityOne", "EntityTwo"]
        entities: list[str] = [e.strip() for e in content.split("<|>") if e.strip()]

        # Store as a JSON array string
        buf.append((json.dumps(entities, ensure_ascii=False), leaf_uuid))
        updated += 1

        # Flush in pages to avoid holding everything in memory
        if len(buf) >= 5_000:
            conn.executemany(
                "UPDATE leafs SET entities = ? WHERE uuid = ?", buf
            )
            conn.commit()
            buf.clear()

    # Flush remainder
    if buf:
        conn.executemany(
            "UPDATE leafs SET entities = ? WHERE uuid = ?", buf
        )
        conn.commit()

    conn.close()
    return updated

In [ ]:
def resubmit_batch(old_batch_id: str) -> str:
    """Resubmit a failed batch from its original .jsonl. Returns new batch_id."""
    metadata = get_metadata()
    entry    = next(m for m in metadata if m["id"] == old_batch_id)

    with open(STAGING_DIR / entry["file_name"], "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    new_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    entry["id"] = new_job.id
    entry["input_file_id"] = batch_file.id
    save_metadata(metadata)
    return new_job.id


In [ ]:
metadata = get_metadata()

# Separate entries that already have a batch job id from fresh ones
submitted   = [m for m in metadata if "id" in m]
unsubmitted = [m for m in metadata if "id" not in m]

print(f"Batches already submitted : {len(submitted):,}")
print(f"Batches to submit         : {len(unsubmitted):,}")
print(f"Total remaining           : {len(metadata):,}\n")

# Process unsubmitted first, then poll any previously-submitted
queue = unsubmitted + submitted

for entry in queue:
    file_name: str     = entry["file_name"]
    input_file_id: str = entry["input_file_id"]
    retries: int       = 0

    # Submit if no batch id yet
    if "id" not in entry:
        batch_job = openai_client.batches.create(
            input_file_id=input_file_id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
        )
        batch_id: str = batch_job.id
        entry["id"] = batch_id
        save_metadata(get_metadata())   # persist the id immediately
        print(f"{file_name} | submitted: {batch_id}")
    else:
        batch_id = entry["id"]
        print(f"{file_name} | resuming poll for: {batch_id}")

    # Poll loop
    while True:
        job = openai_client.batches.retrieve(batch_id)
        status = job.status

        if status == "completed":
            output = openai_client.files.content(job.output_file_id)
            updated = write_entities_to_db(output.text)
            remove_from_metadata(batch_id)

            # Clean up the local .jsonl and remote file
            (STAGING_DIR / file_name).unlink(missing_ok=True)
            try:
                openai_client.files.delete(input_file_id)
            except Exception:
                pass
            try:
                openai_client.files.delete(job.output_file_id)
            except Exception:
                pass

            remaining = len(get_metadata())
            print(f"✓ {file_name} — {updated:,} leafs updated | {remaining} batches remaining\n")
            break

        elif status == "failed":
            errors = job.errors.data if job.errors else []
            token_limit = any(
                "enqueued token limit" in (e.message or "").lower()
                for e in errors
            )
            if token_limit and retries < MAX_RETRIES:
                retries += 1
                print(f"  Token limit hit — waiting {RETRY_WAIT}s then resubmitting "
                      f"(attempt {retries}/{MAX_RETRIES})")
                time.sleep(RETRY_WAIT)
                batch_id = resubmit_batch(batch_id)
                print(f"  Resubmitted as {batch_id}")
                continue
            elif retries >= MAX_RETRIES:
                print(f"  Max retries ({MAX_RETRIES}) reached — skipping {file_name}.\n")
                remove_from_metadata(batch_id)
                break
            else:
                messages = [e.message for e in errors]
                print(f"  Failed: {messages} — skipping {file_name}.\n")
                remove_from_metadata(batch_id)
                break

        elif status in ("cancelling", "cancelled", "expired"):
            print(f"  Status '{status}' — skipping {file_name}.\n")
            remove_from_metadata(batch_id)
            break

        else:
            completed = job.request_counts.completed
            total     = job.request_counts.total
            clear_output(wait=True)
            print(f"{file_name} | {batch_id}")
            print(f"  [{status}] {completed}/{total} — next check in {POLL_INTERVAL}s...")
            time.sleep(POLL_INTERVAL)

print("All batches processed.")

In [ ]:
conn = sqlite3.connect(str(DB_PATH))

total:  int = conn.execute("SELECT COUNT(*) FROM leafs").fetchone()[0]
filled: int = conn.execute("SELECT COUNT(*) FROM leafs WHERE entities IS NOT NULL").fetchone()[0]
empty:  int = total - filled

print(f"Total leafs       : {total:,}")
print(f"With entities     : {filled:,}")
print(f"Still pending     : {empty:,}")
print(f"Completion        : {filled / total * 100:.2f}%")

# Sample
row = conn.execute(
    "SELECT uuid, entities FROM leafs WHERE entities IS NOT NULL LIMIT 1"
).fetchone()
if row:
    print(f"\nSample: {row[0]}")
    print(f"  Entities: {row[1][:200]}...")

conn.close()

In [ ]:
pattern = re.compile(r"^entity_batch_\d+\.jsonl$")

# Delete local .jsonl files
deleted_local = [
    f for f in STAGING_DIR.iterdir()
    if f.is_file() and pattern.match(f.name) and not f.unlink()
]
print(f"Deleted locally  : {len(deleted_local):,} .jsonl files")

# Cancel remote jobs & delete remote files
deleted_files:  int = 0
cancelled_jobs: int = 0

try:
    metadata = get_metadata()
    for entry in metadata:
        file_id = entry.get("input_file_id")
        if file_id:
            try:
                openai_client.files.delete(file_id)
                deleted_files += 1
                print(f"  Deleted file    : {file_id}  ({entry['file_name']})")
            except Exception as e:
                print(f"  Could not delete {file_id}: {e}")

        batch_id = entry.get("id")
        if batch_id:
            try:
                openai_client.batches.cancel(batch_id)
                cancelled_jobs += 1
                print(f"  Cancelled batch : {batch_id}")
            except Exception as e:
                print(f"  Could not cancel {batch_id}: {e}")
except FileNotFoundError:
    print("  No batch_metadata.json — skipping remote cleanup.")

if METADATA_PATH.exists():
    METADATA_PATH.unlink()
    print("Deleted           : batch_metadata.json")

print(f"Deleted files     : {deleted_files:,}")
print(f"Cancelled jobs    : {cancelled_jobs:,}")